# Edge-Waste: Stage 1 Training on Colab

**Workflow:**
1. Mount Google Drive
2. Clone the repo (or pull latest)
3. Install the package
4. Download datasets → Drive
5. Run ingest + split
6. Train (saves checkpoint to Drive)
7. Evaluate

**After training:** share `runs/stage1/best.pt` + `runs/stage1/metrics_test.json` + `runs/stage1/confusion_test.png`

## Step 0 — Mount Drive & check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
print('CUDA:', torch.version.cuda)

import shutil
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive free: {free / 1e9:.1f} GB')

## Step 1 — Clone/pull the repo into Drive

In [ ]:
import os

REPO_DIR = '/content/drive/MyDrive/edge-waste'

if os.path.exists(REPO_DIR):
    print('Repo already exists — pulling latest...')
    !git -C {REPO_DIR} pull
else:
    print('Cloning repo...')
    !git clone https://github.com/darkhorse0204/edge-waste {REPO_DIR}

os.chdir(REPO_DIR)
!pwd
!ls

## Step 2 — Install the package

In [ ]:
os.chdir(REPO_DIR)
!pip install -e '.[detect,data]' -q

# Verify
import sys
sys.path.insert(0, f'{REPO_DIR}/src')
!python -m edgewaste.taxonomy

## Step 3 — Set up Kaggle credentials

Upload your `kaggle.json` to Drive at `MyDrive/kaggle.json`, then run:

In [ ]:
import os, shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

# Copy kaggle.json from Drive to the expected location
shutil.copy('/content/drive/MyDrive/kaggle.json',
            os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle credentials set up.')

# Quick test
!kaggle datasets list -s garbage 2>&1 | head -5

## Step 4 — Download datasets

**Skip this cell if `data/downloads/` already exists on Drive from a previous run.**

In [ ]:
os.chdir(REPO_DIR)

# Check what's already downloaded
import pathlib
downloads = pathlib.Path('data/downloads')
if downloads.exists():
    for src in downloads.iterdir():
        imgs = list(src.rglob('*.jpg')) + list(src.rglob('*.png'))
        print(f'  {src.name}: {len(imgs)} images already present')

# Download missing ones
!edgewaste-fetch --config configs/stage1.yaml

## Step 5 — Ingest + Split

In [ ]:
os.chdir(REPO_DIR)
!edgewaste-ingest --config configs/stage1.yaml
!edgewaste-split --config configs/stage1.yaml

# Show split stats
import pandas as pd
df = pd.read_csv('data/splits.csv')
print('\nSplit distribution:')
print(df.groupby(['split', 'label']).size().unstack(fill_value=0))
print(f'\nTotal images: {len(df)}')

## Step 6 — Smoke Test (fast, ~2 min)

In [ ]:
os.chdir(REPO_DIR)
!edgewaste-train --config configs/stage1.yaml --smoke
print('\n✅ Smoke test passed — pipeline is wired correctly.')

## Step 7 — Full Training

Expected time on Colab T4 GPU: **~20–40 minutes** for 15 epochs.
Checkpoint saved to `runs/stage1/best.pt` (on Drive).

In [ ]:
os.chdir(REPO_DIR)
!edgewaste-train --config configs/stage1.yaml

## Step 8 — Evaluate on Test Split

In [ ]:
os.chdir(REPO_DIR)
!edgewaste-eval --config configs/stage1.yaml --ckpt runs/stage1/best.pt

# Display confusion matrix
from IPython.display import Image as IPImage
IPImage('runs/stage1/confusion_test.png')

## Step 9 — Print summary of artifacts to share

In [ ]:
import json, pathlib

metrics_path = pathlib.Path('runs/stage1/metrics_test.json')
if metrics_path.exists():
    m = json.loads(metrics_path.read_text())
    print('=== TEST RESULTS ===')
    print(f'Accuracy:         {m["accuracy"]*100:.2f}%')
    print(f'Macro F1:         {m["macro"]["f1"]*100:.2f}%')
    print(f'Weighted F1:      {m["weighted"]["f1"]*100:.2f}%')
    print(f'Num test samples: {m["num_samples"]}')
    print(f'Classes:          {m["classes_present"]}')

print('\n=== FILES TO SHARE ===')
for f in ['runs/stage1/best.pt', 'runs/stage1/history.json',
          'runs/stage1/metrics_test.json', 'runs/stage1/confusion_test.png']:
    p = pathlib.Path(f)
    if p.exists():
        size = p.stat().st_size / 1e6
        print(f'  ✅ {f}  ({size:.1f} MB)')
    else:
        print(f'  ❌ {f}  (not found)')

## Step 10 — (Optional) Train Detector too

Only run this after Step 8 passes with ≥80% accuracy.

In [ ]:
os.chdir(REPO_DIR)
!edgewaste-detect-fetch --config configs/detect.yaml
!edgewaste-detect-train --config configs/detect.yaml

# Artifacts
import pathlib
det_ckpt = pathlib.Path('runs/detect/taco_single_class/weights/best.pt')
if det_ckpt.exists():
    print(f'\n✅ Detector checkpoint: {det_ckpt}  ({det_ckpt.stat().st_size/1e6:.1f} MB)')